In [1]:
import asyncio
import time
import json
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, handoff
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX

load_dotenv(override=True)
MODEL_NAME = "gpt-5.4-mini"
print("모델:", MODEL_NAME)


python-dotenv could not parse statement starting at line 2
python-dotenv could not parse statement starting at line 3


모델: gpt-5.4-mini


In [2]:
with open("advisories.json", encoding="utf-8") as f:
    advisories = json.load(f)

adv = next(a for a in advisories if a["no"] == 1)   # 1번 = Log4Shell
advisory_text = adv["advisory"]

print(f"[{adv['no']}] {adv['label']} / {adv['cve_id']}")
print(f"본문 {len(advisory_text)}자\n")
print(advisory_text[:300], "...")


[1] Apache Log4j2 / CVE-2021-44228
본문 2500자

# Summary Log4j versions prior to 2.16.0 are subject to a remote code execution vulnerability via the ldap JNDI parser. As per [Apache's Log4j security guide](https://logging.apache.org/log4j/2.x/security.html): Apache Log4j2 <=2.14.1 JNDI features used in configuration, log messages, and parameters ...


In [3]:
intro = """당신은 보안 공지를 분석하는 전문가입니다.
입력으로 보안 공지(advisory) 본문 하나를 받습니다.
반드시 공지 본문에 있는 내용만 쓰세요. 공지에 없는 것(버전, 수치 등)을 지어내지 마세요.
한국어로 간결하게 답하세요.
"""

summarizer = Agent(
    name="요약가",
    instructions=intro + "이 공지가 어떤 취약점인지 3~4문장으로 요약하세요.",
    model=MODEL_NAME,
)
impact_rater = Agent(
    name="영향도 평가자",
    instructions=intro + "이 취약점의 영향도(심각도, 공격 난이도, 영향 범위)를 3~4문장으로 평가하세요.",
    model=MODEL_NAME,
)
action_writer = Agent(
    name="조치 작성자",
    instructions=intro + "담당자가 바로 실행할 조치를 3~4개 항목으로 쓰세요.",
    model=MODEL_NAME,
)

for a in [summarizer, impact_rater, action_writer]:
    print(f"{a.name:12} 모델={a.model}")


요약가          모델=gpt-5.4-mini
영향도 평가자      모델=gpt-5.4-mini
조치 작성자       모델=gpt-5.4-mini


In [4]:
async def make_card_by_code(text):
    """방식 ①: 파이썬이 전문가 셋을 동시에 부르고, 파이썬이 카드를 조립한다."""
    with trace("방식1-코드로엮기"):
        results = await asyncio.gather(
            Runner.run(summarizer, text),
            Runner.run(impact_rater, text),
            Runner.run(action_writer, text),
        )

    summary, impact, action = [r.final_output for r in results]

    card = f"""# 보안 공지 카드 — {adv['label']} ({adv['cve_id']})

## 요약
{summary}

## 영향도
{impact}

## 조치
{action}
"""
    return card, results


card1, results1 = await make_card_by_code(advisory_text)
print(card1)


# 보안 공지 카드 — Apache Log4j2 (CVE-2021-44228)

## 요약
Apache Log4j에서 JNDI LDAP 파서를 통해 원격 코드 실행(RCE)이 발생할 수 있는 취약점입니다. 공격자가 로그 메시지나 파라미터를 제어할 수 있고 메시지 lookup substitution이 활성화되어 있으면, LDAP 서버에서 로드된 코드를 실행할 수 있습니다. 영향받는 것은 주로 `org.apache.logging.log4j:log4j-core`이며, Log4j 2.15.0 이전 버전이 대상입니다. 2.16.0에서는 JNDI가 기본적으로 비활성화되고 메시지 lookup 지원이 제거되었습니다.

## 영향도
심각도는 매우 높습니다. 공지에 따르면 사용자 제어 로그 메시지나 파라미터를 통해 LDAP/JNDI를 악용해 원격 코드 실행(RCE)이 가능하므로, 영향은 치명적입니다. 공격 난이도는 높지 않은 편으로, 공격자가 로그에 들어가는 입력을 통제할 수 있으면 악용될 수 있습니다. 영향 범위는 `log4j-core`가 포함된 취약한 Log4j 버전 전반이며, 특히 `2.15.0` 이전 버전과 관련해 광범위한 시스템에 영향을 줄 수 있습니다.

## 조치
- `org.apache.logging.log4j:log4j-core`를 **2.16.0 이상**으로 업데이트하세요.
- `log4j-api`도 사용 중이면 **log4j-core와 같은 버전**으로 맞추세요.
- 즉시 반영이 어렵다면, **취약한 Log4j 버전(prior to 2.15.0)** 사용 여부를 점검하고 가능한 범위에서 **2.3.1 / 2.12.2 / 2.12.3** 백포트 버전 적용을 검토하세요.
- 사용자 입력이 로그에 포함되는 경로(에러, 예외 추적, 인증 실패 등)를 점검하고, **untrusted/user-controlled data 로깅**을 최소화하세요.



In [5]:
# 에이전트를 도구로. tool_name은 영문만! (한글 넣으면 BadRequestError)
summarizer_tool = summarizer.as_tool(
    tool_name="summarizer",
    tool_description="보안 공지 본문을 받아 어떤 취약점인지 요약한다.")
impact_tool = impact_rater.as_tool(
    tool_name="impact_rater",
    tool_description="보안 공지 본문을 받아 영향도(심각도, 공격 난이도, 영향 범위)를 평가한다.")
action_tool = action_writer.as_tool(
    tool_name="action_writer",
    tool_description="보안 공지 본문을 받아 담당자가 실행할 조치를 작성한다.")

manager_instructions = """당신은 보안 공지 카드를 만드는 관리자입니다.
세 전문가 도구를 사용해 요약, 영향도, 조치를 각각 받아
하나의 보안 공지 카드로 정리하세요.
각 전문가에게는 받은 공지 본문을 그대로 전달하세요.
카드는 '요약', '영향도', '조치' 세 부분으로 구성하세요.
"""

manager = Agent(
    name="카드 관리자",
    instructions=manager_instructions,
    tools=[summarizer_tool, impact_tool, action_tool],
    model=MODEL_NAME,
)
print("관리자 준비 완료 — 도구", len(manager.tools), "개")


관리자 준비 완료 — 도구 3 개


In [6]:
with trace("방식2-에이전트를도구로"):
    result2 = await Runner.run(manager, advisory_text)

print(result2.final_output)

# 전문가를 몇 번 불렀는지 확인 (openai_agent_code 36셀 방식)
calls = [i.raw_item.name for i in result2.new_items if i.type == "tool_call_item"]
print("\n전문가 호출 순서:", calls)
print("호출 횟수:", len(calls), "(3이면 지시대로)")


**요약**  
Apache Log4j의 `log4j-core`에서 발생하는 RCE 취약점입니다. LDAP/JNDI를 통해 공격자가 로그 메시지나 파라미터를 제어하면 임의 코드 실행이 가능합니다. 2.16.0부터 기본적으로 차단되며, 2.15.0은 일부 상황에서 미흡합니다.

**영향도**  
심각도: 매우 높음  
공격 난이도: 낮음~보통  
영향 범위: 취약한 Log4j 사용 애플리케이션 전반, 특히 사용자 입력이 로그에 들어가는 경우

**조치**  
1. `log4j-core`를 2.16.0 이상으로 즉시 업데이트  
2. `log4j-api`도 동일 버전으로 맞춤  
3. 불가피하면 2.12.3 / 2.12.2 / 2.3.1 백포트 적용 검토  
4. 예외 추적, 인증 실패 등 사용자 입력이 로그에 남는 경로 점검

전문가 호출 순서: ['summarizer', 'impact_rater', 'action_writer']
호출 횟수: 3 (3이면 지시대로)


In [7]:
# 재료를 넘겨받아 카드로 조립하는 에이전트. name은 영문! (transfer_to_<이름> 도구가 자동 생성됨)
card_assembler = Agent(
    name="card_assembler",
    handoff_description="세 전문가의 요약/영향도/조치가 모이면 카드로 조립하는 에이전트",
    instructions=RECOMMENDED_PROMPT_PREFIX + """
당신은 보안 공지 카드를 조립하는 담당자입니다. 관리자에게서 업무를 넘겨받습니다.
지금까지의 대화에 있는 요약, 영향도, 조치 내용을 모아
'요약', '영향도', '조치' 세 부분으로 된 카드 한 장으로 정리하세요.
한국어로 작성하세요.
""",
    model=MODEL_NAME,
)


In [8]:
handoff_manager = Agent(
    name="핸드오프 관리자",
    instructions="""당신은 보안 공지 카드 제작 관리자입니다.
세 전문가 도구로 요약, 영향도, 조치를 모으세요.
셋이 다 모이면 카드 조립 담당에게 넘겨서 카드를 완성하게 하세요.
""",
    tools=[summarizer_tool, impact_tool, action_tool],   # 조립 도구는 없음
    handoffs=[card_assembler],                            # 조립은 넘겨서 처리
    model=MODEL_NAME,
)
print("도구", len(handoff_manager.tools), "개 / 핸드오프", len(handoff_manager.handoffs), "개")


도구 3 개 / 핸드오프 1 개


In [9]:
with trace("방식3-핸드오프"):
    result3 = await Runner.run(handoff_manager, advisory_text)

print(result3.final_output)

# 핸드오프 확인 — 두 가지 다 봐야 함 (안 보면 -10점)
kinds = [i.type for i in result3.new_items]
print("\n핸드오프 발생:", "handoff_output_item" in kinds)
print("마지막 답한 에이전트:", result3.last_agent.name)


**요약**  
Apache Log4j의 JNDI/LDAP 처리 취약점으로, 취약 버전에서 공격자가 제어한 로그 메시지나 파라미터를 통해 원격 코드 실행(RCE)이 발생할 수 있습니다. 특히 `log4j-core`가 직접 영향받으며, 2.16.0부터는 JNDI가 기본 비활성화되고 message lookup도 제거되었습니다.

**영향도**  
심각도가 매우 높습니다. 신뢰할 수 없는 입력(예: 예외 추적, 인증 실패)이 로그에 기록되는 환경에서 임의 코드 실행으로 이어질 수 있습니다. `org.apache.logging.log4j:log4j-core` 사용 시 영향이 있으며, `log4j-api`는 함께 같은 버전으로 맞춰야 합니다.

**조치**  
- Log4j Core를 **2.16.0 이상**으로 즉시 업그레이드  
- `log4j-core`와 `log4j-api` 버전을 **동일하게 유지**  
- **2.15.0만 적용된 경우 추가 점검** 필요  
- 로그에 들어가는 **사용자 제어 입력 경로**를 점검하고 제한

핸드오프 발생: True
마지막 답한 에이전트: card_assembler


In [10]:
def build_card(summary, impact, action):
    return f"""# 보안 공지 카드 — {adv['label']} ({adv['cve_id']})

## 요약
{summary}

## 영향도
{impact}

## 조치
{action}
"""

async def method_code(text):
    """방식① 코드로 엮기"""
    with trace("방식1-코드"):
        results = await asyncio.gather(
            Runner.run(summarizer, text),
            Runner.run(impact_rater, text),
            Runner.run(action_writer, text),
        )
    s, i, a = [r.final_output for r in results]
    return build_card(s, i, a), results, len(results)   # 전문가 직접 3번 호출

async def method_tool(text):
    """방식② 에이전트를 도구로"""
    with trace("방식2-도구"):
        r = await Runner.run(manager, text)
    experts = len([i for i in r.new_items if i.type == "tool_call_item"])
    return r.final_output, [r], experts

async def method_handoff(text):
    """방식③ 핸드오프"""
    with trace("방식3-핸드오프"):
        r = await Runner.run(handoff_manager, text)
    experts = len([i for i in r.new_items if i.type == "tool_call_item"])
    return r.final_output, [r], experts


In [11]:
def usage_correct(results):
    """정답: context_wrapper.usage — 하위 전문가 토큰까지 합산됨"""
    inp = out = req = 0
    for r in results:
        u = r.context_wrapper.usage
        inp += u.input_tokens
        out += u.output_tokens
        req += u.requests          # LLM 호출 횟수 (하위 포함)
    return inp, out, req

def usage_raw_trap(results):
    """함정: raw_responses만 셈 — ②③에서 전문가 토큰이 빠진다"""
    inp = out = 0
    for r in results:
        for resp in r.raw_responses:
            if resp.usage:
                inp += resp.usage.input_tokens
                out += resp.usage.output_tokens
    return inp, out

async def measure(method, text):
    t0 = time.time()
    card, results, experts = await method(text)
    secs = time.time() - t0
    inp, out, req = usage_correct(results)
    rinp, rout = usage_raw_trap(results)
    return {
        "전문가호출": experts, "LLM호출": req,
        "입력토큰": inp, "출력토큰": out, "총토큰": inp + out,
        "총토큰_함정": rinp + rout,      # raw_responses로만 센 값 (비교용)
        "초": round(secs, 2), "글자수": len(card),
    }


In [12]:
import statistics

REPEATS = 3
methods = {"① 코드": method_code, "② 도구": method_tool, "③ 핸드오프": method_handoff}

records = {}
for name, method in methods.items():
    runs = []
    for k in range(REPEATS):
        runs.append(await measure(method, advisory_text))
        print(f"{name}  {k + 1}/{REPEATS} 완료")
    records[name] = runs
print("\n측정 끝")


① 코드  1/3 완료
① 코드  2/3 완료
① 코드  3/3 완료
② 도구  1/3 완료
② 도구  2/3 완료
② 도구  3/3 완료
③ 핸드오프  1/3 완료
③ 핸드오프  2/3 완료
③ 핸드오프  3/3 완료

측정 끝


In [13]:
def cell(runs, key):
    vals = [r[key] for r in runs]
    return f"{statistics.mean(vals):.0f} ({min(vals)}~{max(vals)})"

keys = ["전문가호출", "LLM호출", "입력토큰", "출력토큰", "총토큰", "초", "글자수"]

print(f"{'항목':<10}" + "".join(f"{n:<18}" for n in records))
print("-" * 64)
for key in keys:
    print(f"{key:<10}" + "".join(f"{cell(records[n], key):<18}" for n in records))

# 함정 확인: 정답(context_wrapper) vs 함정(raw_responses만)
print("\n[ 토큰 세는 곳 함정 — 총토큰 ]")
for n in records:
    t  = statistics.mean([r["총토큰"]     for r in records[n]])
    tr = statistics.mean([r["총토큰_함정"] for r in records[n]])
    print(f"  {n}: 정답 {t:.0f}  /  함정(raw만) {tr:.0f}  →  누락 {t - tr:.0f}")


항목        ① 코드              ② 도구              ③ 핸드오프            
----------------------------------------------------------------
전문가호출     3 (3~3)           3 (3~3)           3 (3~3)           
LLM호출     3 (3~3)           6 (5~7)           6 (6~6)           
입력토큰      2115 (2115~2115)  7464 (6129~10121) 7101 (6259~8101)  
출력토큰      449 (435~463)     2555 (2540~2574)  1841 (1569~2147)  
총토큰       2564 (2550~2578)  10019 (8669~12671)8942 (7828~10248) 
초         2 (2.38~2.44)     16 (13.48~18.75)  13 (10.93~14.51)  
글자수       908 (889~932)     542 (507~577)     525 (481~569)     

[ 토큰 세는 곳 함정 — 총토큰 ]
  ① 코드: 정답 2564  /  함정(raw만) 2564  →  누락 0
  ② 도구: 정답 10019  /  함정(raw만) 7498  →  누락 2521
  ③ 핸드오프: 정답 8942  /  함정(raw만) 7131  →  누락 1811


## 결론

1. 매일 같은 순서로 도는 작업이면 **① 코드로 엮기**를 쓰겠다. 총토큰이 2564로 ②(10019)·③(8942)의 약 1/4 수준이고, 순서가 `asyncio.gather`로 보장되어 예측 가능하기 때문이다.

2. "전문가 3회 호출"은 세 방식 모두 지켜졌다(전문가호출 3회). 다만 지시가 지켜졌어도 **토큰과 LLM 호출 수는 실행마다 흔들렸다.** 즉 "호출 횟수가 지켜졌다"가 "모든 것이 안정적이다"를 뜻하지는 않는다.

3. 토큰은 ②가 ①의 약 **3.9배**(10019 vs 2564)였다. 관리자가 공지 본문을 전문가 셋에게 각각 다시 넘기고, 전문가의 응답을 또 입력으로 받기 때문에 **입력 토큰**에서 차이가 커진다.

4. ①은 `asyncio.gather`로 셋을 **동시에** 부르고, ②③은 관리자가 **차례로** 부른다. 그래서 ①이 빠른 데에는 "일이 적어서"와 "동시에 해서"가 섞여 있어, 순수한 방식 속도 비교로는 공정하지 않다.

### 함정 메모
`raw_responses`로만 토큰을 세면 ②③에서 각각 약 2521, 1811 토큰이 누락된다. `as_tool`로 부른 전문가가 자기 안에서 부른 LLM 사용량이 `raw_responses`에는 안 잡히기 때문이다. 그래서 `context_wrapper.usage`로 세야 한다.
